#  Generative Adversarial Network (GAN)

Implementar un modelo de GANs (generación de imágenes): usar images de la CNN, caras pequeñas como CelebA,  para entrenar un DCGAN. 

### Objetivo: 
- Entrenar una un modelo usando el dataset CelebA.

### Puntos Clave: 
- Generar una GAN estable usando tecnicas aprendidas.
- Genearar una cuadricula de imagenes generadas para ver como progresan la calidad al avanzar las epocas.
- Realizar una grafica de la perdida del Discriminador vs la perdida del Generador.

In [1]:
# DCGAN para generación de rostros con CelebA

## Objetivo

# Implementar una red generativa adversarial convolucional profunda, o DCGAN, para generar imágenes sintéticas de rostros humanos a partir del dataset CelebA.

## Entregables del módulo GAN

# 1. Carga y preprocesamiento del dataset CelebA.
# 2. Definición del Generador.
# 3. Definición del Discriminador.
# 4. Definición de funciones de pérdida.
# 5. Implementación del paso de entrenamiento con `GradientTape`.
# 6. Entrenamiento alternado de Generador y Discriminador.
# 7. Guardado de imágenes generadas por época.
# 8. Cuadrícula de evolución visual de imágenes generadas.
# 9. Gráfica de pérdida del Discriminador vs pérdida del Generador.
# 10. Comentario técnico de resultados para reporte o README.

## Descripción técnica breve
# Una GAN está formada por dos redes neuronales que compiten entre sí. El Generador recibe un vector aleatorio `z`
#  del espacio latente y produce una imagen sintética. El Discriminador recibe imágenes reales y generadas, y aprende
#  a distinguirlas. En una DCGAN, ambas redes utilizan capas convolucionales para trabajar directamente con la estructura espacial de las imágenes.

In [2]:
!pip install tensorflow tensorflow-datasets matplotlib pandas numpy pillow imageio

In [3]:
import os
import time
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("GPU disponible:", tf.config.list_physical_devices("GPU"))

# Reproducibilidad parcial
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Directorios de salida
PROJECT_DIR = Path(".")
RESULTS_DIR = PROJECT_DIR / "results"
IMAGES_DIR = RESULTS_DIR / "generated_images"
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"
MODELS_DIR = RESULTS_DIR / "models"

for folder in [RESULTS_DIR, IMAGES_DIR, CHECKPOINT_DIR, MODELS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Carpetas creadas correctamente.")

TensorFlow: 2.20.0
GPU disponible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Carpetas creadas correctamente.


In [4]:
# Tamaño de imagen de trabajo
IMG_SIZE = 64
CHANNELS = 3
IMAGE_SHAPE = (IMG_SIZE, IMG_SIZE, CHANNELS)

# Vector latente z
LATENT_DIM = 100

# Entrenamiento
BATCH_SIZE = 64
EPOCHS = 30

# Para pruebas rápidas, podés usar 10000 o 30000 imágenes.
# Para usar todo CelebA, poné MAX_IMAGES = None.
MAX_IMAGES = 30000

# Learning rates
GEN_LR = 2e-4
DISC_LR = 1e-4

# Parámetros Adam recomendados en DCGAN
BETA_1 = 0.5
BETA_2 = 0.999

# Ruido fijo para evaluar la evolución visual en cada época
NUM_EXAMPLES_TO_GENERATE = 16
FIXED_NOISE = tf.random.normal([NUM_EXAMPLES_TO_GENERATE, LATENT_DIM])

print("Configuración cargada.")

Configuración cargada.


In [ ]:
def find_local_celeba_dir():
    """
    Busca automáticamente una carpeta que contenga imágenes .jpg de CelebA.
    
    Rutas frecuentes:
    - Kaggle: /kaggle/input/celeba-dataset/img_align_celeba/img_align_celeba
    - Local: ./data/celeba/img_align_celeba
    - Local: ./data/img_align_celeba
    """
    candidate_dirs = [
        "/kaggle/input/celeba-dataset/img_align_celeba/img_align_celeba",
        "/kaggle/input/celeba-dataset/img_align_celeba",
        "./data/celeba/img_align_celeba",
        "./data/img_align_celeba",
        "./celeba/img_align_celeba",
        "./img_align_celeba",
    ]
    
    for d in candidate_dirs:
        path = Path(d)
        if path.exists():
            jpg_files = list(path.glob("*.jpg"))
            if len(jpg_files) > 0:
                return path
    
    return None


def preprocess_image_from_path(path):
    """
    Lee una imagen desde disco, la recorta al centro, la redimensiona
    y la normaliza al rango [-1, 1].
    
    La normalización [-1, 1] se usa porque el Generador terminará con activación tanh.
    """
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.cast(image, tf.float32)
    
    # CelebA alineado suele venir aproximadamente en 218x178.
    # Se recorta/paddea al centro para obtener una región cuadrada.
    image = tf.image.resize_with_crop_or_pad(image, 178, 178)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    
    # Normalización de [0, 255] a [-1, 1]
    image = (image / 127.5) - 1.0
    return image


def preprocess_image_from_tfds(example):
    """
    Preprocesamiento equivalente para ejemplos cargados desde tensorflow_datasets.
    """
    image = example["image"]
    image = tf.cast(image, tf.float32)
    image = tf.image.resize_with_crop_or_pad(image, 178, 178)
    image = tf.image.resize(image, [IMG_SIZE, IMG_SIZE])
    image = (image / 127.5) - 1.0
    return image


celeba_dir = find_local_celeba_dir()

if celeba_dir is not None:
    print(f"Dataset local encontrado en: {celeba_dir}")
    
    image_paths = tf.data.Dataset.list_files(
        str(celeba_dir / "*.jpg"),
        shuffle=True,
        seed=SEED
    )
    
    if MAX_IMAGES is not None:
        image_paths = image_paths.take(MAX_IMAGES)
    
    train_dataset = (
        image_paths
        .map(preprocess_image_from_path, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(10000, seed=SEED)
        .batch(BATCH_SIZE, drop_remainder=True)
        .prefetch(tf.data.AUTOTUNE)
    )

else:
    print("No se encontró CelebA local. Intentando cargar con tensorflow_datasets...")
    import tensorflow_datasets as tfds
    
    raw_ds = tfds.load(
        "celeb_a",
        split="train",
        shuffle_files=True
    )
    
    if MAX_IMAGES is not None:
        raw_ds = raw_ds.take(MAX_IMAGES)
    
    train_dataset = (
        raw_ds
        .map(preprocess_image_from_tfds, num_parallel_calls=tf.data.AUTOTUNE)
        .shuffle(10000, seed=SEED)
        .batch(BATCH_SIZE, drop_remainder=True)
        .prefetch(tf.data.AUTOTUNE)
    )

print("Dataset preparado.")

No se encontró CelebA local. Intentando cargar con tensorflow_datasets...


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]